[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_LanguageDetection.ipynb)

# Benchmark: Language Detection

Scores a pretrained language identifier's accuracy against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="languagedetection")`.

**Dataset**: a small hand-picked multilingual sample (~4 sentences per language, across the
languages `ld_wiki_tatoeba_cnn_21` was trained on). The model itself was trained on
Wikipedia + [Tatoeba](https://tatoeba.org/) -- for a larger benchmark, pull a bigger sample
directly from Tatoeba's per-language-pair sentence exports.

**Model**: `LanguageDetectorDL.pretrained()` (default: `ld_wiki_tatoeba_cnn_21`).

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import LanguageDetectorDL
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
rows = [
    ("The quick brown fox jumps over the lazy dog near the river.", "en"),
    ("Machine learning models can process large amounts of text data.", "en"),
    ("The history of Rome spans more than two thousand years.", "en"),
    ("She walked to the market to buy fresh vegetables and bread.", "en"),
    ("Le renard brun rapide saute par-dessus le chien paresseux.", "fr"),
    ("La cuisine francaise est reputee dans le monde entier.", "fr"),
    ("Paris est la capitale de la France depuis des siecles.", "fr"),
    ("Les enfants jouaient dans le parc pendant tout l'apres-midi.", "fr"),
    ("Der schnelle braune Fuchs springt ueber den faulen Hund.", "de"),
    ("Die deutsche Geschichte ist reich an kulturellen Ereignissen.", "de"),
    ("Berlin ist die Hauptstadt und groesste Stadt Deutschlands.", "de"),
    ("Viele Touristen besuchen jedes Jahr die bayerischen Alpen.", "de"),
    ("El rapido zorro marron salta sobre el perro perezoso.", "es"),
    ("La historia de Espana esta llena de momentos fascinantes.", "es"),
    ("Madrid es la capital y la ciudad mas grande de Espana.", "es"),
    ("Los ninos jugaban felices en el parque durante la tarde.", "es"),
    ("La volpe marrone veloce salta sopra il cane pigro.", "it"),
    ("La cucina italiana e famosa in tutto il mondo.", "it"),
    ("Roma e la capitale d'Italia da oltre due millenni.", "it"),
    ("I bambini giocavano felici nel parco durante il pomeriggio.", "it"),
    ("A raposa marrom rapida pula sobre o cachorro preguicoso.", "pt"),
    ("A historia do Brasil comeca com a chegada dos portugueses.", "pt"),
    ("Lisboa e a capital e a maior cidade de Portugal.", "pt"),
    ("As criancas brincavam felizes no parque durante a tarde.", "pt"),
    ("De snelle bruine vos springt over de luie hond.", "nl"),
    ("Amsterdam is de hoofdstad en grootste stad van Nederland.", "nl"),
    ("De geschiedenis van Nederland gaat eeuwen terug in de tijd.", "nl"),
    ("De kinderen speelden de hele middag vrolijk in het park.", "nl"),
]

gold_data = spark.createDataFrame(rows, ["text", "label"])
gold_data.groupBy("label").count().orderBy("label").show()

+-----+-----+
|label|count|
+-----+-----+
|   de|    4|
|   en|    4|
|   es|    4|
|   fr|    4|
|   it|    4|
|   nl|    4|
|   pt|    4|
+-----+-----+

## 2. Build the pipeline

In [10]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
language_detector = LanguageDetectorDL.pretrained() \
    .setInputCols(["document"]).setOutputCol("language")

pipeline = Pipeline(stages=[document_assembler, language_detector])
pipeline_model = pipeline.fit(gold_data)

ld_wiki_tatoeba_cnn_21 download started this may take some time.
Approximate size to download 7.1 MB

[ | ]
[ / ]
[OK!]

## 3. Run the benchmark

In [12]:
report = Benchmark.evaluate(pipeline_model, gold_data, task="languagedetection", label_col="label")
print(report)

languagedetection accuracy (n=28): accuracy=0.9643, weightedF1=0.9637, weightedPrecision=0.9714, weightedRecall=0.9643
  de: f1=1.0000, precision=1.0000, recall=1.0000
  en: f1=1.0000, precision=1.0000, recall=1.0000
  es: f1=1.0000, precision=1.0000, recall=1.0000
  fr: f1=0.8889, precision=0.8000, recall=1.0000
  it: f1=1.0000, precision=1.0000, recall=1.0000
  nl: f1=0.8571, precision=1.0000, recall=0.7500
  pt: f1=1.0000, precision=1.0000, recall=1.0000